# Qa ray casting

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA: Ray Casting Directional Geometry

This notebook validates localized directional feature extraction from `src/ray_caster.py` by overlaying selected starburst rays on satellite imagery and inspecting profile statistics.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

try:
    import contextily as ctx
except ImportError:
    ctx = None
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

plt.rcParams["figure.dpi"] = 120
TARGET_EPSG = 32633


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "geometric_builder")
        candidates.append(parent)

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if all((candidate / name).exists() for name in ("configs", "data", "notebooks", "src")):
            return candidate
    raise FileNotFoundError("Could not resolve project root with configs/data/notebooks/src.")


def load_bathy(nc_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[float]]:
    with xr.open_dataset(nc_path) as ds:
        if "z" in ds.data_vars:
            da = ds["z"]
        elif "depth" in ds.data_vars:
            da = ds["depth"]
        else:
            raise KeyError("Expected bathymetry variable `z` or `depth`.")

        x = np.asarray(da["x"].to_numpy(), dtype=float)
        y = np.asarray(da["y"].to_numpy(), dtype=float)
        z = np.asarray(da.transpose("y", "x").to_numpy(), dtype=float)

    extent = [float(np.nanmin(x)), float(np.nanmax(x)), float(np.nanmin(y)), float(np.nanmax(y))]
    return x, y, z, extent


PROJECT_ROOT = resolve_project_root()
SUBGRID_NC = PROJECT_ROOT / "data/bathy/subgrid_padded.nc"
RAY_CSV = PROJECT_ROOT.parent / "data/processed/ray_features.csv"

x_vals, y_vals, z_grid, full_extent = load_bathy(SUBGRID_NC)
ray_df = pd.read_csv(RAY_CSV)

for col in ["distance_profile_m", "elevation_profile_m", "gradient_profile", "laplacian_profile"]:
    ray_df[col] = ray_df[col].fillna("[]").apply(json.loads)

print(f"Project root: {PROJECT_ROOT}")
print(f"Target CRS: EPSG:{TARGET_EPSG}")
print(f"Grid shape (y, x): {z_grid.shape}")
print(f"Ray feature rows: {len(ray_df)}")
print(f"Sites: {ray_df['site_name'].nunique()}")

In [ ]:
RANDOM_SEED = 7  # Change this to get a different reproducible set of 6 sites. Use None for non-reproducible sampling.

selected_count = min(6, ray_df["site_name"].nunique())
if selected_count == 0:
    raise ValueError("No sites found in ray_features.csv")

site_groups = {
    site_name: group.copy() for site_name, group in ray_df.groupby("site_name", sort=False)
}
site_records = []

for site_name, site_df in site_groups.items():
    site_x = float(site_df["site_x"].iloc[0])
    site_y = float(site_df["site_y"].iloc[0])
    if not (
        full_extent[0] <= site_x <= full_extent[1] and full_extent[2] <= site_y <= full_extent[3]
    ):
        continue

    dx = np.abs(site_df["endpoint_x"].to_numpy(dtype=float) - site_x)
    dy = np.abs(site_df["endpoint_y"].to_numpy(dtype=float) - site_y)
    max_dx = float(np.nanmax(dx)) if dx.size else 0.0
    max_dy = float(np.nanmax(dy)) if dy.size else 0.0
    x_pad = max(75.0, 0.03 * max_dx)
    y_pad = max(75.0, 0.03 * max_dy)
    half_width = max_dx + x_pad
    half_height = max_dy + y_pad

    site_records.append(
        {
            "site_name": site_name,
            "site_x": site_x,
            "site_y": site_y,
            "half_width": half_width,
            "half_height": half_height,
        }
    )

site_catalog = pd.DataFrame(site_records)
if site_catalog.empty:
    raise ValueError(
        "No in-domain sites found in ray_features.csv for the current bathymetry extent"
    )

site_catalog["cap_half_width"] = np.minimum(
    site_catalog["site_x"] - full_extent[0], full_extent[1] - site_catalog["site_x"]
)
site_catalog["cap_half_height"] = np.minimum(
    site_catalog["site_y"] - full_extent[2], full_extent[3] - site_catalog["site_y"]
)
site_catalog = site_catalog.loc[
    (site_catalog["half_width"] <= site_catalog["cap_half_width"])
    & (site_catalog["half_height"] <= site_catalog["cap_half_height"])
].copy()
if len(site_catalog) < selected_count:
    raise ValueError(
        "Fewer than 6 sites can be centered inside their own ray windows for the current bathymetry extent"
    )

min_height_to_width_ratio = 0.75
unique_widths = np.sort(site_catalog["half_width"].unique())
unique_heights = np.sort(site_catalog["half_height"].unique())
best_pool = None
common_half_width = None
common_half_height = None
best_area = None

for trial_half_width in unique_widths:
    width_pool = site_catalog.loc[
        (site_catalog["half_width"] <= trial_half_width)
        & (site_catalog["cap_half_width"] >= trial_half_width)
    ].copy()
    if len(width_pool) < selected_count:
        continue

    for trial_half_height in unique_heights:
        trial_half_height = max(
            float(trial_half_height), float(min_height_to_width_ratio * trial_half_width)
        )
        pool = width_pool.loc[
            (width_pool["half_height"] <= trial_half_height)
            & (width_pool["cap_half_height"] >= trial_half_height)
        ].copy()
        if len(pool) < selected_count:
            continue

        area = float(trial_half_width * trial_half_height)
        if best_area is None or area < best_area:
            best_area = area
            best_pool = pool.copy()
            common_half_width = float(trial_half_width)
            common_half_height = float(trial_half_height)
        break

if best_pool is None:
    raise ValueError(
        "Could not construct a shared centered window for 6 sites within the current bathymetry extent"
    )

sample_kwargs = {} if RANDOM_SEED is None else {"random_state": RANDOM_SEED}
selection = best_pool.sample(n=selected_count, replace=False, **sample_kwargs)
selected_sites = selection["site_name"].tolist()
selected_site_data = {site_name: site_groups[site_name] for site_name in selected_sites}
print(f"Using RANDOM_SEED={RANDOM_SEED!r}; selected sites: {selected_sites}")

z_masked = np.ma.masked_invalid(z_grid)

fig, axes = plt.subplots(2, 3, figsize=(15, 15), squeeze=False)
axes = axes.ravel()

for ax, site_name in zip(axes, selected_sites):
    site_df = selected_site_data[site_name]
    site_x = float(site_df["site_x"].iloc[0])
    site_y = float(site_df["site_y"].iloc[0])

    xmin = site_x - common_half_width
    xmax = site_x + common_half_width
    ymin = site_y - common_half_height
    ymax = site_y + common_half_height

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    if ctx is not None:
        try:
            ctx.add_basemap(
                ax,
                source=ctx.providers.Esri.WorldImagery,
                crs="EPSG:32633",
                zoom=9,
                attribution=False,
                reset_extent=False,
            )
        except Exception as exc:
            print(f"Basemap fetch failed ({exc}); plotting without imagery.")

    ax.imshow(
        z_masked,
        extent=full_extent,
        origin="lower",
        cmap="Blues_r",
        alpha=0.08,
        interpolation="nearest",
        zorder=1,
    )
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, pos, sx=site_x: f"{(x - sx) / 1000:+.1f}")
    )
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda y, pos, sy=site_y: f"{(y - sy) / 1000:+.1f}")
    )

    for _, ray in site_df.iterrows():
        ax.plot(
            [ray["site_x"], ray["endpoint_x"]],
            [ray["site_y"], ray["endpoint_y"]],
            color="#00d4ff",
            linewidth=1.0,
            alpha=0.65,
            zorder=2,
        )

    ax.scatter(site_x, site_y, s=16, c="yellow", edgecolor="black", zorder=3)
    ax.set_title(f"{site_name}")
    ax.set_xlabel("X Offset From Site (km)")
    ax.set_ylabel("Y Offset From Site (km)")

for ax in axes[len(selected_sites) :]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
profile_summary = ray_df[
    [
        "site_name",
        "angle_deg",
        "fetch_m",
        "max_slope",
        "mean_laplacian",
        "max_abs_laplacian",
        "hit_land",
    ]
].copy()

styled = (
    profile_summary.sort_values(["site_name", "angle_deg"])
    .style.format(
        {
            "angle_deg": "{:.0f}",
            "fetch_m": "{:.1f}",
            "max_slope": "{:.5f}",
            "mean_laplacian": "{:.6f}",
            "max_abs_laplacian": "{:.6f}",
        },
        na_rep="NaN",
    )
    .background_gradient(subset=["max_slope"], cmap="YlOrRd")
    .background_gradient(subset=["mean_laplacian", "max_abs_laplacian"], cmap="coolwarm")
)

display(styled)
ray_df.head()